# Multi-Document Agents

- https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents
- https://developers.llamaindex.ai/python/examples/agent/multi_document_agents-v1/

In this guide, you learn towards setting up an agent that can effectively answer different types of questions over a larger set of documents.

These questions include the following

- QA over a specific doc
- QA comparing different docs
- Summaries over a specific doc
- Comparing summaries between different docs

We do this with the following architecture:

- setup a "document agent" over each Document: each doc agent can do QA/summarization within its doc
- setup a top-level agent over this set of document agents. Do tool retrieval and then do CoT over the set of tools to answer a question.

## Setup and Download Data

In this section, we'll define imports and then download Wikipedia articles about different cities. Each article is stored separately.

We load in 18 cities - this is not quite at the level of "hundreds" of documents but its still large enough to warrant some top-level document retrieval!

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-agent-openai
%pip install llama-index-embeddings-openai
%pip install llama-index-llms-openai

In [ ]:
!pip install llama-index

In [10]:
!ls

CHANGELOG.md   __init__.py    data           poetry.lock
Dockerfile     __pycache__    fastapi_app.py pyproject.toml
Makefile       app.py         models         src
README.md      assets         notebooks      tests


In [1]:
import os

os.chdir("/Users/josingh/hobby/climate-dashboard/server")

In [2]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleKeywordTableIndex,
    SimpleDirectoryReader,
)
from llama_index.core import SummaryIndex
from llama_index.core.schema import IndexNode
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.openai import OpenAI
from llama_index.core.callbacks import CallbackManager

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
ndc_file_name_path_mapping = {
    'Cambodia': 'data/ndc/Cambodia/2025- Cambodia-NDC 3.0_0.pdf', 
    'Myanmar': 'data/ndc/Myanmar/Myanmar Updated  NDC July 2021.pdf', 
    'Laos': 'data/ndc/Laos/NDC 2020 of Lao PDR (English), 09 April 2021 (1).pdf', 
    'Singapore': 'data/ndc/Singapore/Singapore_Second_Nationally_Determined_Contribution.pdf',
    'Brunei': "data/ndc/Brunei/[Final] Brunei Darussalam's Nationally Determined Contribution 3.0 2025.pdf", 
    'Vietnam': 'data/ndc/Vietnam/Viet Nam NDC 2022 Update.pdf', 
    'Malaysia': 'data/ndc/Malaysia/Malaysia NDC 3.0 to UNFCCC 2025 final.pdf', 
    'Indonesia': 'data/ndc/Indonesia/Indonesia_Second NDC_2025.10.24.pdf',
    'Thailand': 'data/ndc/Thailand/TH NDC 3.0.pdf',
    "Philippines": "data/ndc/Philippines/Philippines - NDC.pdf",
}

sea_countries = list(ndc_file_name_path_mapping.keys())

In [4]:
sea_countries

['Cambodia',
 'Myanmar',
 'Laos',
 'Singapore',
 'Brunei',
 'Vietnam',
 'Malaysia',
 'Indonesia',
 'Thailand',
 'Philippines']

In [5]:
# Load all country documents
country_docs = {}
for country in sea_countries:
    country_docs[country] = SimpleDirectoryReader(
        input_files=[ndc_file_name_path_mapping[country]]
    ).load_data()

Define Global LLM and Embeddings

In [6]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.llm = OpenAI(temperature=0, model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

## Building Multi-Document Agents

In this section we show you how to construct the multi-document agent. We first build a document agent for each document, and then define the top-level parent agent with an object index.

### Build Document Agent for each Document

In this section we define "document agents" for each document.

We define both a vector index (for semantic search) and summary index (for summarization) for each document. The two query engines are then converted into tools that are passed to an OpenAI function calling agent.

This document agent can dynamically choose to perform semantic search or summarization within a given document.

We create a separate document agent for each city.

In [7]:
from llama_index.agent.openai import OpenAIAgent
from llama_index.core import load_index_from_storage, StorageContext
from llama_index.core.node_parser import SentenceSplitter
import os

node_parser = SentenceSplitter()

# Build agents dictionary
agents = {}
query_engines = {}

# this is for the baseline
all_nodes = []

for idx, country in enumerate(sea_countries):
    nodes = node_parser.get_nodes_from_documents(country_docs[country])
    all_nodes.extend(nodes)

    if not os.path.exists(f"./data/vector_store/ndc/{country}"):
        # build vector index
        vector_index = VectorStoreIndex(nodes)
        vector_index.storage_context.persist(
            persist_dir=f"./data/vector_store/ndc/{country}"
        )
    else:
        vector_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=f"./data/vector_store/ndc/{country}"),
        )

    # build summary index
    summary_index = SummaryIndex(nodes)
    # define query engines
    vector_query_engine = vector_index.as_query_engine(llm=Settings.llm)
    summary_query_engine = summary_index.as_query_engine(llm=Settings.llm)

    # define tools
    query_engine_tools = [
        QueryEngineTool(
            query_engine=vector_query_engine,
            metadata=ToolMetadata(
                name="vector_tool",
                description=(
                    "Useful for questions related to specific aspects of"
                    f" {country}."
                ),
            ),
        ),
        QueryEngineTool(
            query_engine=summary_query_engine,
            metadata=ToolMetadata(
                name="summary_tool",
                description=(
                    "Useful for any requests that require a holistic summary"
                    f" of EVERYTHING about {country}. For questions about"
                    " more specific sections, please use the vector_tool."
                ),
            ),
        ),
    ]

    # build agent
    function_llm = OpenAI(model="gpt-4o-mini")
    agent = OpenAIAgent.from_tools(
        query_engine_tools,
        llm=function_llm,
        verbose=True,
        system_prompt=f"""\
You are a specialized agent designed to answer queries about {country}.
You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on prior knowledge.\
""",
    )

    agents[country] = agent
    query_engines[country] = vector_index.as_query_engine(
        similarity_top_k=2
    )

### Build Retriever-Enabled OpenAI Agent

We build a top-level agent that can orchestrate across the different document agents to answer any user query.

This agent takes in all document agents as tools. This specific agent `RetrieverOpenAIAgent` performs tool retrieval before tool use (unlike a default agent that tries to put all tools in the prompt).

Here we use a top-k retriever, but we encourage you to customize the tool retriever method!


In [8]:
# define tool for each document agent
all_tools = []
for country in sea_countries:
    country_summary = (
        f"This content contains Nationally Determined Contributions for {country}. Use"
        f" this tool if you want to answer any questions about {country}.\n"
    )
    doc_tool = QueryEngineTool(
        query_engine=agents[country],
        metadata=ToolMetadata(
            name=f"tool_{country}",
            description=country_summary,
        ),
    )
    all_tools.append(doc_tool)

In [9]:
# define an "object" index and retriever over these tools
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    all_tools,
    index_cls=VectorStoreIndex,
)

In [10]:
from llama_index.agent.openai import OpenAIAgent

top_agent = OpenAIAgent.from_tools(
    tool_retriever=obj_index.as_retriever(similarity_top_k=3),
    system_prompt=""" \
You are an agent designed to answer queries about a set of given South East Asia countries.
Please always use the tools provided to answer a question. Do not rely on prior knowledge.\

""",
    verbose=True,
)

## Running Example Queries

Let's run some example queries, ranging from QA / summaries over a single document to QA / summarization over multiple documents.

In [11]:
# should use Boston agent -> vector tool
response = top_agent.query("Tell me about the NDCs of Singapore")

Added user message to memory: Tell me about the NDCs of Singapore
=== Calling Function ===
Calling function: tool_Singapore with args: {"input":"Nationally Determined Contributions"}
Added user message to memory: Nationally Determined Contributions
=== Calling Function ===
Calling function: vector_tool with args: {"input":"Nationally Determined Contributions Singapore"}
Got output: Singapore's Nationally Determined Contributions (NDC) reflect its commitment to addressing climate change despite significant challenges. The country plans to utilize internationally transferred mitigation outcomes (ITMOs) under Article 6 of the Paris Agreement to help meet its emissions reduction targets. Given its unique national circumstances, including limited land area, high population density, and a lack of natural resources, Singapore faces difficulties in large-scale deployment of renewable energy sources. 

The nation recognizes that its small and open economy, along with its energy-disadvantaged st

In [12]:
print(response)

Singapore's Nationally Determined Contributions (NDC) reflect its commitment to addressing climate change, despite facing significant challenges. The country plans to utilize internationally transferred mitigation outcomes (ITMOs) under Article 6 of the Paris Agreement to meet its emissions reduction targets.

Due to its unique circumstances—such as limited land area, high population density, and a lack of natural resources—Singapore faces difficulties in deploying large-scale renewable energy sources. The nation recognizes that its small and open economy, along with its energy-disadvantaged status, necessitates international partnerships for effective decarbonization.

Singapore actively engages in global climate action and promotes regional and multilateral cooperation. Although its contribution to global emissions is minimal, the country is particularly vulnerable to the impacts of climate change, which requires comprehensive adaptation strategies.

In summary, Singapore's NDC empha

In [13]:
# should use Houston agent -> vector tool
response = top_agent.query(
    "Give me a summary of all positive aspects of Singapore's NDC"
)

Added user message to memory: Give me a summary of all positive aspects of Singapore's NDC
=== Calling Function ===
Calling function: tool_Singapore with args: {"input":"positive aspects"}
Added user message to memory: positive aspects
=== Calling Function ===
Calling function: summary_tool with args: {"input":"positive aspects of Singapore"}
Got output: Singapore demonstrates a strong commitment to sustainable development and climate action, prioritizing a balance between economic growth and environmental stewardship. The country has made significant strides in reducing emissions, transitioning from fuel oil to natural gas for power generation, which now constitutes about 95% of its fuel mix. 

Additionally, Singapore has implemented a carbon pricing scheme, becoming the first in Southeast Asia to do so, and has plans to further increase the carbon tax, making it one of the highest in Asia. The nation is actively pursuing innovative solutions for renewable energy, such as maximizing s

In [14]:
print(response)

Singapore's Nationally Determined Contributions (NDC) highlight several positive aspects, particularly in its commitment to sustainable development and climate action:

1. **Sustainable Development**: Singapore effectively balances economic growth with environmental stewardship, making significant progress in reducing emissions.

2. **Energy Transition**: The country has shifted from fuel oil to natural gas for power generation, with natural gas now making up about 95% of its fuel mix.

3. **Carbon Pricing**: Singapore is the first in Southeast Asia to implement a carbon pricing scheme and plans to increase the carbon tax, establishing itself as a leader in climate policy in the region.

4. **Innovative Renewable Solutions**: Despite land constraints, Singapore maximizes solar energy deployment and explores low-carbon electricity imports.

5. **Stakeholder Engagement**: The government engages comprehensively with various sectors to contribute to decarbonization efforts.

6. **Investmen

In [15]:
response = top_agent.query(
    "Tell me the NDCs of Singapore, and then compare that with the"
    " NDCs of Malaysia"
)

Added user message to memory: Tell me the NDCs of Singapore, and then compare that with the NDCs of Malaysia
=== Calling Function ===
Calling function: tool_Singapore with args: {"input": "Nationally Determined Contributions of Singapore"}
Added user message to memory: Nationally Determined Contributions of Singapore
=== Calling Function ===
Calling function: vector_tool with args: {"input":"Nationally Determined Contributions of Singapore"}
Got output: Singapore's Nationally Determined Contributions (NDCs) reflect its commitment to addressing climate change while considering its unique national circumstances. The country plans to utilize internationally transferred mitigation outcomes (ITMOs) under Article 6 of the Paris Agreement to help meet its emissions reduction targets. This approach is necessary due to Singapore's challenges in decarbonization, stemming from its small land area, high population density, and limited renewable energy resources.

The nation recognizes that its dom

In [16]:
print(response)

### Nationally Determined Contributions (NDCs) Overview

#### Singapore's NDCs:
1. **International Cooperation**: Singapore aims to utilize internationally transferred mitigation outcomes (ITMOs) under Article 6 of the Paris Agreement to meet its emissions reduction targets, emphasizing the need for global collaboration due to limited domestic abatement potential.
2. **Challenges in Decarbonization**: The country faces significant challenges due to its small land area, high population density, and limited renewable energy resources.
3. **Vulnerability to Climate Change**: Despite contributing only about 0.1% of global emissions, Singapore is particularly vulnerable to climate change impacts, necessitating comprehensive adaptation strategies that will incur significant costs.
4. **Regional and Multilateral Partnerships**: Singapore actively engages in global climate action efforts and advocates for partnerships to enhance its decarbonization initiatives.

#### Malaysia's NDCs:
1. **NDC 

In [17]:
response = top_agent.query(
    "Tell me the differences between Malaysia's and Indonesia's NDCs in terms of"
    " meeting their climate targets."
)

Added user message to memory: Tell me the differences between Malaysia's and Indonesia's NDCs in terms of meeting their climate targets.
=== Calling Function ===
Calling function: tool_Malaysia with args: {"input": "What are the Nationally Determined Contributions (NDCs) of Malaysia regarding climate targets?"}
Added user message to memory: What are the Nationally Determined Contributions (NDCs) of Malaysia regarding climate targets?
=== Calling Function ===
Calling function: vector_tool with args: {"input":"Nationally Determined Contributions (NDCs) of Malaysia regarding climate targets"}
Got output: Malaysia's Nationally Determined Contributions (NDCs) outline its climate targets, which include a commitment to peak national greenhouse gas emissions between 2029 and 2034. The country aims for an absolute reduction of 15 to 30 million tonnes of CO₂ equivalent (MtCO₂eq) by 2035 from the peak level. This target includes an unconditional reduction of up to 20 MtCO₂eq, with an additional 1

In [18]:
print(str(response))

The Nationally Determined Contributions (NDCs) of Malaysia and Indonesia reflect their respective commitments to meeting climate targets, but they differ significantly in scope, ambition, and specific goals. Here are the key differences:

### Malaysia's NDCs:
- **Peak Emissions**: Malaysia aims to peak its national greenhouse gas emissions between 2029 and 2034.
- **Reduction Targets**: The country targets an absolute reduction of 15 to 30 million tonnes of CO₂ equivalent (MtCO₂eq) by 2035 from the peak level, with:
  - An unconditional reduction of up to 20 MtCO₂eq.
  - An additional reduction of 10 MtCO₂eq contingent upon receiving international support (climate finance, technology transfer, and capacity-building).
- **Coverage**: Malaysia's NDC encompasses all seven greenhouse gases, including CO₂, methane, nitrous oxide, and various fluorinated gases.

### Indonesia's NDCs:
- **Initial Commitment**: Indonesia's Intended NDC aimed for a 29% reduction in greenhouse gas emissions by 2